# 1. Simulation

This notebook contains the complete simulation section for the paper. All experiment settings and training configurations are defined **here**; the notebook only calls reusable functions from `countflow`.

We use six comparison rows:

1. Count Flow Map;
2. Count-FM with the original unit-jump sampler;
3. Count-FM with binomial $\tau$-leaping;
4. CountsDiff;
5. D3PM with an ordinal Gaussian transition matrix;
6. Discrete Flow Maps.

The two Count-FM rows share one trained Count-FM checkpoint. The five learned models use the same formal training budget (10,000 updates), effective batch size 512, and hidden width/depth 256/4 where the method permits.

The notebook has two experimental parts:

- **Part 1:** exact 2D benchmark, including endpoint quality and intermediate count distributions;
- **Part 2:** $d=32$, high-count benchmark;

For cluster runs, `scripts/run_simulation.py` writes to the same output directories. With `RESUME=True`, this notebook reuses completed script results and performs the paper-facing analysis/plots.

In [ ]:
from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "countflow").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from countflow.benchmark_data import benchmark_settings, decode_categories
from countflow.bridge import sample_signed_binomial_bridge
from countflow.model import CountFlowMap
from countflow.sampling import generate_count_samples
from countflow.simulation_runner import run_simulation_suite, METHOD_LABELS
from countflow.plotting import plot_sample_panels
from countflow.baselines import (
    CountRateModel,
    CountsDiffModel,
    D3PMModel,
    DiscreteFlowMapModel,
    sample_original_unit_jump,
    sample_binomial_tau_leap,
    sample_countsdiff,
    sample_d3pm,
    sample_simplex_flow_map,
)
from countflow.baselines.countsdiff import survival_probability, randomized_round
from countflow.utils import set_seed

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TORCH_DEVICE = torch.device(DEVICE)
print("device:", DEVICE)


## Shared paper configuration


In [ ]:
# Formal paper settings. Change TRAIN_STEPS here for short tests or the final run.
SEEDS = [42, 123, 2026, 7, 31415]
NFE_VALUES = [1, 2, 4, 16, 64, 128, 256]

# Plotting-only choices. These DO NOT change the formal run/config hash.
# Choose any subset of NFE_VALUES for exploratory endpoint sample panels.
SAMPLE_NFE_VALUES = list(NFE_VALUES)  # e.g. [1, 4, 256] or [2, 16, 128]
SAMPLE_PLOT_N = 5_000

# Paper-facing operating points. Both lists can be changed without retraining.
PAPER_TABLE_NFE_VALUES = [1, 4, 128, 256]
PAPER_SAMPLE_NFE_VALUES = [1, 4, 128, 256]

# Intermediate visualization uses normalized generation progress q in [0, 1].
# Flow Maps use exactly INTERMEDIATE_FLOW_NFE finite-time compositions.
# Count-FM / CountsDiff / D3PM use their fine/native 256-step paths.
INTERMEDIATE_PROGRESS_VALUES = [0.0, 0.25, 0.50, 0.75, 1.0]
INTERMEDIATE_PLOT_N = 2_500
INTERMEDIATE_FLOW_NFE = 4
INTERMEDIATE_NATIVE_STEPS = 256

TRAIN_STEPS = 10_000
BATCH_SIZE = 512
HIDDEN_DIM = 256
DEPTH = 4
TAU = 0.98
RESUME = True  # trains if results are missing; loads completed script results when the config matches

# Step-dependent schedules.
# At TRAIN_STEPS=10_000:
#   CK warmup = 200
#   span warmup = 1,000
#   Discrete Flow Map = 8,000 + 2,000
CK_WARMUP_STEPS = max(1, int(0.02 * TRAIN_STEPS))
SPAN_WARMUP_STEPS = max(1, int(0.10 * TRAIN_STEPS))

DFM_DIAGONAL_STEPS = max(1, int(0.80 * TRAIN_STEPS))
DFM_DISTILLATION_STEPS = TRAIN_STEPS - DFM_DIAGONAL_STEPS

# Gives reasonably frequent progress information for short tests,
# while giving log_every=200 for the 10k formal run.
LOG_EVERY = max(1, TRAIN_STEPS // 50)

METHODS = [
    "count_flow_map",
    "count_fm_unit_jump",
    "count_fm_binomial_tau_leap",
    "countsdiff",
    "d3pm",
    "discrete_flow_map",
]

# Formal target/source definitions.
BENCHMARKS = benchmark_settings("paper")

COMMON_MODELS = {
    "count_flow_map": {
        "steps": TRAIN_STEPS,
        "batch_size": BATCH_SIZE,
        "learning_rate": 1e-3,
        "hidden_dim": HIDDEN_DIM,
        "depth": DEPTH,
        "n_mixtures": 8,
        "ck_weight": 1.0,
        "ck_warmup_steps": CK_WARMUP_STEPS,
        "span_warmup_steps": SPAN_WARMUP_STEPS,
        "log_every": LOG_EVERY,
    },

    "count_fm": {
        "steps": TRAIN_STEPS,
        "batch_size": BATCH_SIZE,
        "learning_rate": 1e-3,
        "hidden_dim": HIDDEN_DIM,
        "depth": DEPTH,
        "weight_decay": 0.0,
        "ema_decay": 0.999,
        "log_every": LOG_EVERY,
    },

    "countsdiff": {
        "steps": TRAIN_STEPS,
        "batch_size": BATCH_SIZE,
        "learning_rate": 1e-3,
        "hidden_dim": HIDDEN_DIM,
        "depth": DEPTH,
        "weight_decay": 0.0,
        "ema_decay": 0.999,
        "eta_rescale": 0.0,
        "log_every": LOG_EVERY,
    },

    "d3pm": {
        "steps": TRAIN_STEPS,
        "batch_size": BATCH_SIZE,
        "learning_rate": 1e-3,
        "hidden_dim": HIDDEN_DIM,
        "depth": DEPTH,
        "weight_decay": 0.0,
        "ema_decay": 0.999,
        "diffusion_steps": 256,
        "beta_start": 1e-4,
        "beta_end": 0.02,
        "rescale_betas": True,
        "transition_bands": None,
        "auxiliary_weight": 0.001,
        "log_every": LOG_EVERY,
    },

    "discrete_flow_map": {
        "steps": TRAIN_STEPS,
        "batch_size": BATCH_SIZE,
        "learning_rate": 1e-3,
        "hidden_dim": HIDDEN_DIM,
        "depth": DEPTH,
        "weight_decay": 0.0,
        "ema_decay": 0.999,
        "consistency_weight": 1.0,
        "prior": "gaussian",
        "diagonal_steps": DFM_DIAGONAL_STEPS,
        "distillation_steps": DFM_DISTILLATION_STEPS,
        "adaptive_r": 0.5,
        "adaptive_c": 0.01,
        "gradient_surgery": True,
        "log_every": LOG_EVERY,
    },
}

METRICS = {
    "mmd_max_points": 5000,
    "w2_max_points": 1500,
    "w2_repeats": 3,
    "sliced_w2_projections": 128,
    "sliced_w2_max_points": 10000,
}


def make_config(setting_name):
    return {
        "preset": "paper",
        "settings": [setting_name],
        "seeds": list(SEEDS),
        "methods": list(METHODS),
        "nfe_values": list(NFE_VALUES),
        "tau": TAU,

        # Used for the 32D / 128D experiments.
        "generated_samples": 5_000,
        "target_samples": 5_000,

        # More samples for the exact 2D experiment because TV is estimated
        # directly from the generated probability mass.
        "exact_2d_generated_samples": 30_000,
        "exact_2d_target_samples": 30_000,

        # Used only to determine finite support for categorical methods.
        "categorical_pilot_samples": 200_000,
        "categorical_tail_probability": 1e-4,

        "timing_repeats": 3,
        "intermediate_times": [0.25, 0.50, 0.75, 0.98] if setting_name == "exact_2d" else [],
        "intermediate_samples": 5_000,
        "intermediate_count_fm_steps": 256,
        "models": {name: dict(values) for name, values in COMMON_MODELS.items()},
        "metric_settings": dict(METRICS),
    }


def aggregate_metrics(frame, metric):
    return (
        frame.groupby(["method_label", "nfe"])[metric]
        .agg(["mean", "std"])
        .reset_index()
    )


def plot_quality_curves(frame, exact_2d):
    metric_names = ["tv", "w2", "mmd2_rbf"] if exact_2d else ["sliced_w2", "mmd2_rbf"]
    titles = ["TV to target", "$W_2$", "MMD$^2_{RBF}$"] if exact_2d else ["Sliced $W_2$", "MMD$^2_{RBF}$"]

    for metric, title in zip(metric_names, titles):
        fig, ax = plt.subplots(figsize=(7.2, 4.4))

        for label, group in frame.groupby("method_label"):
            summary = group.groupby("nfe")[metric].mean().sort_index()
            ax.plot(summary.index, summary.values, marker="o", label=label)

        ax.set_xscale("log", base=2)
        ax.set_xticks(NFE_VALUES, labels=[str(v) for v in NFE_VALUES])
        ax.set_xlabel("NFE")
        ax.set_ylabel(title)
        ax.legend(fontsize=8)

        fig.tight_layout()
        plt.show()


def plot_runtime_curve(frame, metric):
    fig, ax = plt.subplots(figsize=(7.2, 4.4))

    for label, group in frame.groupby("method_label"):
        summary = (
            group.groupby("nfe")[["runtime_seconds", metric]]
            .mean()
            .sort_index()
        )
        ax.plot(
            summary["runtime_seconds"],
            summary[metric],
            marker="o",
            label=label,
        )

    ax.set_xscale("log")
    ax.set_xlabel("generation time (seconds)")
    ax.set_ylabel(metric)
    ax.legend(fontsize=8)

    fig.tight_layout()
    plt.show()


print("training updates:", TRAIN_STEPS)
print("CK warmup:", CK_WARMUP_STEPS)
print("span warmup:", SPAN_WARMUP_STEPS)
print(
    "Discrete Flow Map stages:",
    DFM_DIAGONAL_STEPS,
    "+",
    DFM_DISTILLATION_STEPS,
    "=",
    DFM_DIAGONAL_STEPS + DFM_DISTILLATION_STEPS,
)
print("NFE grid:", NFE_VALUES)

# Part 1 — Exact 2D benchmark

This part provides exact-mass and geometric endpoint evaluation, representative samples, and the learned intermediate count distributions.

**Local paper run:** execute the cell below. Training progress is shown as one concise progress bar per learned model/seed, followed by one evaluation progress bar.

**Cluster mode:** `python scripts/run_simulation.py --part 1 --device cuda`. The notebook reuses the same output directory.

In [ ]:
PART1 = make_config("exact_2d")
part1_output = ROOT / "outputs" / "simulation" / "part1_exact_2d"

metrics_2d = run_simulation_suite(
    ROOT,
    PART1,
    output_dir=part1_output,
    device=DEVICE,
    resume=RESUME,
    progress=True,
)
metrics_2d.head()


In [ ]:
# Full quality curves use every NFE in NFE_VALUES.
summary_2d = (
    metrics_2d.groupby(["method_label", "nfe"])[
        ["tv", "w2", "mmd2_rbf", "mean_relative_l1", "variance_relative_l1",
         "zero_fraction_l1", "correlation_fro_per_dim", "runtime_seconds", "samples_per_second", "parameters"]
    ]
    .agg(["mean", "std"])
)
display(summary_2d)
plot_quality_curves(metrics_2d, exact_2d=True)
plot_runtime_curve(metrics_2d, "w2")


In [ ]:
# Endpoint sample panels are fully user-configurable through SAMPLE_NFE_VALUES.
# If a requested NFE was not saved by the cluster script, this cell generates
# ONLY plotting samples from the existing checkpoint; it never retrains.

invalid_nfe = sorted(set(SAMPLE_NFE_VALUES) - set(NFE_VALUES))
if invalid_nfe:
    raise ValueError(f"SAMPLE_NFE_VALUES must be a subset of NFE_VALUES; invalid={invalid_nfe}")

setting_2d = BENCHMARKS["exact_2d"]
support_path = part1_output / "categorical_support.json"
if not support_path.exists():
    raise FileNotFoundError("categorical_support.json is missing. Run the formal Part 1 script first.")
C_MAX_2D = int(json.loads(support_path.read_text())["exact_2d"]["c_max"])

_PLOT_MODEL_CACHE = {}


def _load_checkpoint_state(model, checkpoint):
    if not checkpoint.exists():
        raise FileNotFoundError(f"Missing checkpoint: {checkpoint}")
    payload = torch.load(checkpoint, map_location=TORCH_DEVICE, weights_only=False)
    state_dict = payload["state_dict"] if isinstance(payload, dict) and "state_dict" in payload else payload
    model.load_state_dict(state_dict)
    return model.to(TORCH_DEVICE).eval()


def _load_plot_model(method, seed):
    canonical = "count_fm" if method.startswith("count_fm_") else method
    key = (canonical, int(seed))
    if key in _PLOT_MODEL_CACHE:
        return _PLOT_MODEL_CACHE[key]

    if canonical == "count_flow_map":
        model_cfg = COMMON_MODELS["count_flow_map"]
        model = CountFlowMap(
            dim=setting_2d.dim,
            hidden_dim=int(model_cfg["hidden_dim"]),
            depth=int(model_cfg["depth"]),
            n_mixtures=int(model_cfg["n_mixtures"]),
            count_scale=float(setting_2d.suggested_count_scale),
        )
        checkpoint = part1_output / "checkpoints" / "exact_2d" / str(seed) / "count_flow_map.pt"
    elif canonical == "count_fm":
        model_cfg = COMMON_MODELS["count_fm"]
        model = CountRateModel(
            dim=setting_2d.dim,
            hidden_dim=int(model_cfg["hidden_dim"]),
            depth=int(model_cfg["depth"]),
            count_scale=float(setting_2d.suggested_count_scale),
        )
        checkpoint = part1_output / "checkpoints" / "exact_2d" / str(seed) / "count_fm.pt"
    elif canonical == "countsdiff":
        model_cfg = COMMON_MODELS["countsdiff"]
        model = CountsDiffModel(
            setting_2d.dim,
            hidden_dim=int(model_cfg["hidden_dim"]),
            depth=int(model_cfg["depth"]),
            count_scale=float(setting_2d.suggested_count_scale),
        )
        checkpoint = part1_output / "checkpoints" / "exact_2d" / str(seed) / "countsdiff.pt"
    elif canonical == "d3pm":
        model_cfg = COMMON_MODELS["d3pm"]
        model = D3PMModel(
            setting_2d.dim,
            C_MAX_2D + 2,
            diffusion_steps=int(model_cfg["diffusion_steps"]),
            hidden_dim=int(model_cfg["hidden_dim"]),
            depth=int(model_cfg["depth"]),
            beta_start=float(model_cfg["beta_start"]),
            beta_end=float(model_cfg["beta_end"]),
            transition_bands=model_cfg.get("transition_bands"),
            rescale_betas=bool(model_cfg.get("rescale_betas", True)),
        )
        checkpoint = part1_output / "checkpoints" / "exact_2d" / str(seed) / "d3pm.pt"
    elif canonical == "discrete_flow_map":
        model_cfg = COMMON_MODELS["discrete_flow_map"]
        model = DiscreteFlowMapModel(
            setting_2d.dim,
            C_MAX_2D + 2,
            hidden_dim=int(model_cfg["hidden_dim"]),
            depth=int(model_cfg["depth"]),
            prior=str(model_cfg["prior"]),
        )
        checkpoint = part1_output / "checkpoints" / "exact_2d" / str(seed) / "discrete_flow_map.pt"
    else:
        raise ValueError(method)

    model = _load_checkpoint_state(model, checkpoint)
    _PLOT_MODEL_CACHE[key] = model
    return model


@torch.no_grad()
def _generate_endpoint_plot_samples(method, nfe, seed, n_samples=SAMPLE_PLOT_N):
    model = _load_plot_model(method, seed)
    set_seed(int(seed) * 100_000 + int(nfe) * 101 + METHODS.index(method))
    if method == "count_flow_map":
        samples, _ = generate_count_samples(
            model, setting_2d.source, n_samples, tau=TAU, n_steps=int(nfe), device=TORCH_DEVICE
        )
        return samples.cpu()
    if method == "count_fm_unit_jump":
        return sample_original_unit_jump(
            model, setting_2d.source, n_samples, int(nfe), tau=TAU, device=DEVICE
        ).cpu()
    if method == "count_fm_binomial_tau_leap":
        return sample_binomial_tau_leap(
            model, setting_2d.source, n_samples, int(nfe), tau=TAU, device=DEVICE, death_rule="linear"
        ).cpu()
    if method == "countsdiff":
        return sample_countsdiff(
            model, n_samples, int(nfe), device=DEVICE,
            eta_rescale=float(COMMON_MODELS["countsdiff"].get("eta_rescale", 0.0)),
        ).cpu()
    if method == "d3pm":
        return sample_d3pm(model, n_samples, int(nfe), c_max=C_MAX_2D, device=DEVICE).cpu()
    if method == "discrete_flow_map":
        return sample_simplex_flow_map(
            model, n_samples, int(nfe), c_max=C_MAX_2D, device=DEVICE
        ).cpu()
    raise ValueError(method)


def _load_or_generate_endpoint_samples(method, nfe, seed, n_samples=SAMPLE_PLOT_N):
    formal_path = part1_output / "samples" / "exact_2d" / str(seed) / f"{method}_nfe{nfe}.pt"
    if formal_path.exists():
        samples = torch.load(formal_path, map_location="cpu", weights_only=False)
        return torch.as_tensor(samples)[:n_samples]

    cache_dir = part1_output / "notebook_endpoint_samples" / "exact_2d" / str(seed)
    cache_dir.mkdir(parents=True, exist_ok=True)
    cache_path = cache_dir / f"{method}_nfe{nfe}_n{n_samples}.pt"
    if cache_path.exists():
        return torch.load(cache_path, map_location="cpu", weights_only=False)

    print(f"Generating plotting samples from checkpoint: {METHOD_LABELS[method]}, NFE={nfe}")
    samples = _generate_endpoint_plot_samples(method, nfe, seed, n_samples=n_samples)
    torch.save(samples, cache_path)
    return samples


def _common_square_limits(sample_arrays, pad=1.0):
    tensors = [torch.as_tensor(x).float().reshape(-1) for x in sample_arrays]
    lower = min(0.0, min(float(x.min().item()) for x in tensors)) - 0.5
    upper = max(float(x.max().item()) for x in tensors) + float(pad)
    return lower, upper


seed = SEEDS[0]
target_samples = setting_2d.target.sample(SAMPLE_PLOT_N).cpu()

# Load all requested endpoint panels first, then determine ONE axis range shared
# by every method AND every requested NFE.
_endpoint_exploratory = {}
_axis_arrays = [target_samples]
for nfe in SAMPLE_NFE_VALUES:
    _endpoint_exploratory[nfe] = {}
    for method in METHODS:
        samples = _load_or_generate_endpoint_samples(method, nfe, seed, SAMPLE_PLOT_N)
        _endpoint_exploratory[nfe][method] = samples
        _axis_arrays.append(samples)

endpoint_plot_min, endpoint_plot_max = _common_square_limits(_axis_arrays)

for nfe in SAMPLE_NFE_VALUES:
    panels = [target_samples] + [_endpoint_exploratory[nfe][m] for m in METHODS]
    titles = ["target"] + [METHOD_LABELS[m] for m in METHODS]

    fig = plot_sample_panels(
        panels, titles, max_points=2500,
        figure_size=(3.0 * len(panels), 3.4),
    )
    for ax in fig.axes:
        ax.set_xlim(endpoint_plot_min, endpoint_plot_max)
        ax.set_ylim(endpoint_plot_min, endpoint_plot_max)
        ax.set_aspect("equal", adjustable="box")
    fig.suptitle(f"Exact 2D samples, NFE={nfe}")
    fig.tight_layout()
    plt.show()


## Intermediate generation paths — all methods

All methods are shown at the same normalized generation progress \(q\in[0,1]\).

- **Reference bridge:** actual bridge time is \(t=q\tau\), with \(\tau=0.98\).
- **Count Flow Map:** a genuine `INTERMEDIATE_FLOW_NFE`-step composition from \(0\) to \(\tau\); with the default NFE=4, the five columns are the states after 0/1/2/3/4 map evaluations.
- **Discrete Flow Maps:** the same NFE=4 composition over its native interval \(0\to1\); its simplex state is argmax-decoded only for visualization.
- **Count-FM, CountsDiff, D3PM:** their fine/native 256-step trajectories, sampled at matched normalized progress.

Thus the all-method figure is a transparent view of each method's native generation path. Quantitative bridge fidelity is interpreted only for the methods that share the known count bridge.


In [ ]:
# Quantitative bridge fidelity produced by the formal runner.
# This table is meaningful only for Count Flow Map / Count-FM because they share
# the known count bridge. The all-method visualization below shows native paths.
intermediate_path = part1_output / "intermediate_metrics.csv"
if intermediate_path.exists():
    intermediate = pd.read_csv(intermediate_path)
    display(
        intermediate.groupby(["method_label", "time"])[["w2", "mmd2_rbf"]]
        .agg(["mean", "std"])
    )
else:
    print("intermediate_metrics.csv not found; skipping the quantitative bridge table.")


def _validate_intermediate_progress():
    values = [float(q) for q in INTERMEDIATE_PROGRESS_VALUES]
    if values[0] != 0.0 or values[-1] != 1.0 or any(q < 0 or q > 1 for q in values):
        raise ValueError("INTERMEDIATE_PROGRESS_VALUES must start at 0, end at 1, and lie in [0,1].")
    for q in values:
        step = q * INTERMEDIATE_FLOW_NFE
        if abs(step - round(step)) > 1e-8:
            raise ValueError(
                "For Flow Map trajectory panels, each progress value must lie on an "
                f"INTERMEDIATE_FLOW_NFE={INTERMEDIATE_FLOW_NFE} step boundary; got q={q}."
            )
    return values


INTERMEDIATE_PROGRESS_VALUES = _validate_intermediate_progress()


@torch.no_grad()
def _reference_bridge_samples(progress, n_samples):
    actual_t = float(progress) * TAU
    if progress <= 0:
        return setting_2d.source.sample(n_samples).cpu()
    set_seed(910_000 + int(round(progress * 10_000)))
    x0 = setting_2d.source.sample(n_samples, device=TORCH_DEVICE)
    x1 = setting_2d.target.sample(n_samples, device=TORCH_DEVICE)
    t = torch.full((n_samples,), actual_t, device=TORCH_DEVICE)
    return sample_signed_binomial_bridge(x0, x1, t).cpu()


@torch.no_grad()
def _count_flow_map_trajectory(n_samples, seed):
    model = _load_plot_model("count_flow_map", seed)
    set_seed(seed * 10_000 + 101)
    x = setting_2d.source.sample(n_samples, device=TORCH_DEVICE)
    states = [x.cpu()]
    grid = torch.linspace(0.0, TAU, INTERMEDIATE_FLOW_NFE + 1, device=TORCH_DEVICE)
    for k in range(INTERMEDIATE_FLOW_NFE):
        s = grid[k].expand(n_samples)
        t = grid[k + 1].expand(n_samples)
        x = model.sample(x, s, t)
        states.append(x.cpu())
    return states


@torch.no_grad()
def _discrete_flow_map_trajectory(n_samples, seed):
    model = _load_plot_model("discrete_flow_map", seed)
    set_seed(seed * 10_000 + 106)
    x = model.sample_prior(n_samples, TORCH_DEVICE)
    states = [decode_categories(x.argmax(dim=-1), C_MAX_2D).cpu()]
    grid = torch.linspace(0.0, 1.0, INTERMEDIATE_FLOW_NFE + 1, device=TORCH_DEVICE)
    for k in range(INTERMEDIATE_FLOW_NFE):
        s = grid[k].expand(n_samples)
        t = grid[k + 1].expand(n_samples)
        x = model.flow(x, s, t)
        states.append(decode_categories(x.argmax(dim=-1), C_MAX_2D).cpu())
    return states


@torch.no_grad()
def _count_fm_intermediate(method, progress, n_samples, seed):
    if progress <= 0:
        return setting_2d.source.sample(n_samples).cpu()
    model = _load_plot_model(method, seed)
    actual_t = float(progress) * TAU
    local_steps = max(1, int(round(INTERMEDIATE_NATIVE_STEPS * float(progress))))
    set_seed(seed * 10_000 + int(round(progress * 1000)) + (2 if method.endswith("unit_jump") else 3))
    if method == "count_fm_unit_jump":
        return sample_original_unit_jump(
            model, setting_2d.source, n_samples, local_steps,
            tau=actual_t, device=DEVICE,
        ).cpu()
    return sample_binomial_tau_leap(
        model, setting_2d.source, n_samples, local_steps,
        tau=actual_t, device=DEVICE, death_rule="linear",
    ).cpu()


@torch.no_grad()
def _countsdiff_intermediate(progress, n_samples, seed):
    model = _load_plot_model("countsdiff", seed)
    set_seed(seed * 10_000 + int(round(progress * 1000)) + 4)
    x = torch.zeros(n_samples, model.dim, device=TORCH_DEVICE, dtype=torch.long)
    if progress <= 0:
        return x.cpu()

    total_steps = int(INTERMEDIATE_NATIVE_STEPS)
    steps_done = min(total_steps, max(1, int(round(float(progress) * total_steps))))
    grid = torch.linspace(1.0, 0.0, total_steps + 1, device=TORCH_DEVICE)
    eta = float(COMMON_MODELS["countsdiff"].get("eta_rescale", 0.0))
    eps = 1e-8
    for index in range(steps_done):
        t_scalar, s_scalar = grid[index], grid[index + 1]
        t = t_scalar.expand(n_samples)
        predicted_missing = randomized_round(model(x, t))
        p_t = survival_probability(t_scalar).clamp(0.0, 1.0)
        p_s = survival_probability(s_scalar).clamp(0.0, 1.0)
        if float(p_t) <= eps:
            sigma_max = torch.tensor(1.0, device=TORCH_DEVICE)
        else:
            sigma_max = torch.minimum(
                torch.tensor(1.0, device=TORCH_DEVICE),
                (1.0 - p_s) / p_t.clamp_min(eps),
            )
        sigma = (eta * sigma_max).clamp(0.0, sigma_max)
        beta = ((p_s - (1.0 - sigma) * p_t) / (1.0 - p_t).clamp_min(eps)).clamp(0.0, 1.0)
        survivors = torch.binomial(x.float(), torch.full_like(x.float(), float(1.0 - sigma))).long()
        births = torch.binomial(predicted_missing.float(), torch.full_like(predicted_missing.float(), float(beta))).long()
        x = survivors + births
    return x.cpu()


@torch.no_grad()
def _d3pm_intermediate(progress, n_samples, seed):
    model = _load_plot_model("d3pm", seed)
    set_seed(seed * 10_000 + int(round(progress * 1000)) + 5)
    x = torch.randint(0, model.n_categories, (n_samples, model.dim), device=TORCH_DEVICE)
    if progress <= 0:
        return decode_categories(x, C_MAX_2D).cpu()

    total_steps = model.diffusion_steps
    steps_done = min(total_steps, max(1, int(round(float(progress) * total_steps))))
    for index in range(steps_done):
        t_value = total_steps - index
        s_value = t_value - 1
        t_index = torch.full((n_samples,), t_value, device=TORCH_DEVICE, dtype=torch.long)
        p_x0 = torch.softmax(model(x, t_index), dim=-1)
        if s_value == 0:
            x = torch.multinomial(
                p_x0.reshape(-1, model.n_categories), 1
            ).reshape(n_samples, model.dim)
        else:
            posterior = model.posterior_skip(x, p_x0, s=s_value, t=t_value)
            x = torch.multinomial(
                posterior.reshape(-1, model.n_categories), 1
            ).reshape(n_samples, model.dim)
    return decode_categories(x, C_MAX_2D).cpu()


def _build_intermediate_samples(seed, n_samples=INTERMEDIATE_PLOT_N):
    cache_dir = (
        part1_output / "notebook_intermediate_samples_v2" / "exact_2d" / str(seed)
        / f"flow{INTERMEDIATE_FLOW_NFE}_native{INTERMEDIATE_NATIVE_STEPS}_n{n_samples}"
    )
    cache_dir.mkdir(parents=True, exist_ok=True)

    flow_states = _count_flow_map_trajectory(n_samples, seed)
    dfm_states = _discrete_flow_map_trajectory(n_samples, seed)

    out = {"reference": [], "count_flow_map": [], "count_fm_unit_jump": [],
           "count_fm_binomial_tau_leap": [], "countsdiff": [], "d3pm": [],
           "discrete_flow_map": []}

    for progress in INTERMEDIATE_PROGRESS_VALUES:
        step_idx = int(round(progress * INTERMEDIATE_FLOW_NFE))
        for method_key in out:
            tag = f"q{progress:.2f}".replace(".", "p")
            path = cache_dir / f"{method_key}_{tag}.pt"
            if path.exists():
                samples = torch.load(path, map_location="cpu", weights_only=False)
            else:
                if method_key == "reference":
                    samples = _reference_bridge_samples(progress, n_samples)
                elif method_key == "count_flow_map":
                    samples = flow_states[step_idx]
                elif method_key == "discrete_flow_map":
                    samples = dfm_states[step_idx]
                elif method_key == "count_fm_unit_jump":
                    samples = _count_fm_intermediate(method_key, progress, n_samples, seed)
                elif method_key == "count_fm_binomial_tau_leap":
                    samples = _count_fm_intermediate(method_key, progress, n_samples, seed)
                elif method_key == "countsdiff":
                    samples = _countsdiff_intermediate(progress, n_samples, seed)
                elif method_key == "d3pm":
                    samples = _d3pm_intermediate(progress, n_samples, seed)
                torch.save(samples, path)
            out[method_key].append(samples)
    return out


_INTERMEDIATE_LABELS = {
    "reference": "Reference bridge",
    "count_flow_map": f"Count Flow Map (NFE={INTERMEDIATE_FLOW_NFE})",
    "count_fm_unit_jump": f"Count-FM + unit jump ({INTERMEDIATE_NATIVE_STEPS} steps)",
    "count_fm_binomial_tau_leap": f"Count-FM + binomial tau-leap ({INTERMEDIATE_NATIVE_STEPS} steps)",
    "countsdiff": f"CountsDiff ({INTERMEDIATE_NATIVE_STEPS} steps)",
    "d3pm": f"D3PM ({INTERMEDIATE_NATIVE_STEPS} steps)",
    "discrete_flow_map": f"Discrete Flow Maps (NFE={INTERMEDIATE_FLOW_NFE}; argmax simplex)",
}

seed = SEEDS[0]
all_intermediate = _build_intermediate_samples(seed, INTERMEDIATE_PLOT_N)

# Exploratory all-method intermediate figure, with ONE common range.
_intermediate_arrays = [
    torch.as_tensor(x) for row in all_intermediate.values() for x in row
]
intermediate_plot_min, intermediate_plot_max = _common_square_limits(_intermediate_arrays)

row_keys = ["reference"] + METHODS
fig, axes = plt.subplots(
    len(row_keys), len(INTERMEDIATE_PROGRESS_VALUES),
    figsize=(3.0 * len(INTERMEDIATE_PROGRESS_VALUES), 2.65 * len(row_keys)),
    squeeze=False, sharex=True, sharey=True,
)
rng = np.random.default_rng(123)
for r, method_key in enumerate(row_keys):
    for c, progress in enumerate(INTERMEDIATE_PROGRESS_VALUES):
        ax = axes[r, c]
        array = torch.as_tensor(all_intermediate[method_key][c]).cpu().numpy()
        if len(array) > 1800:
            array = array[rng.choice(len(array), 1800, replace=False)]
        ax.scatter(array[:, 0], array[:, 1], s=7, alpha=0.30)
        ax.set_xlim(intermediate_plot_min, intermediate_plot_max)
        ax.set_ylim(intermediate_plot_min, intermediate_plot_max)
        ax.set_aspect("equal", adjustable="box")
        if r == 0:
            ax.set_title(f"q={progress:.2f}")
        if c == 0:
            ax.set_ylabel(_INTERMEDIATE_LABELS[method_key] + "\ncount 2")
        if r == len(row_keys) - 1:
            ax.set_xlabel("count 1")
fig.suptitle(
    "Intermediate/native generation paths (same normalized progress q; native paths need not share the same bridge)",
    y=1.002,
)
fig.tight_layout()
plt.show()


In [ ]:
# PAPER TABLE — exact 2D endpoint quality at selected operating points
paper_dir = part1_output / "paper_outputs"
paper_dir.mkdir(parents=True, exist_ok=True)

invalid = sorted(set(PAPER_TABLE_NFE_VALUES) - set(NFE_VALUES))
if invalid:
    raise ValueError(f"PAPER_TABLE_NFE_VALUES contains invalid NFE values: {invalid}")

_method_order = [METHOD_LABELS[m] for m in METHODS]
_metric_specs = [
    ("tv", "TV ↓", lambda x: f"{x:.3f}"),
    ("w2", r"$W_2$ ↓", lambda x: f"{x:.2f}"),
    ("runtime_seconds", "Time (s) ↓", lambda x: f"{x:.3f}"),
]

paper_table = pd.DataFrame(index=_method_order)
for nfe in PAPER_TABLE_NFE_VALUES:
    block = metrics_2d.loc[metrics_2d["nfe"] == nfe]
    for metric, label, formatter in _metric_specs:
        summary = block.groupby("method_label")[metric].agg(["mean", "std"])
        paper_table[(f"NFE={nfe}", label)] = [
            f"{formatter(summary.loc[name, 'mean'])} ± {formatter(summary.loc[name, 'std'])}"
            for name in _method_order
        ]

paper_table.columns = pd.MultiIndex.from_tuples(paper_table.columns)
paper_table.index.name = "Method"
display(paper_table)

flat_table = paper_table.copy()
flat_table.columns = [f"{a} | {b}" for a, b in flat_table.columns]
flat_table.to_csv(paper_dir / "table1_exact2d.csv")
(paper_dir / "table1_exact2d.tex").write_text(
    paper_table.to_latex(escape=False, multicolumn=True, multirow=True),
    encoding="utf-8",
)
print("saved:", paper_dir / "table1_exact2d.csv")
print("saved:", paper_dir / "table1_exact2d.tex")


In [ ]:
# PAPER FIGURE 1 — endpoint samples, with ONE shared range across all panels
paper_dir = part1_output / "paper_outputs"
paper_dir.mkdir(parents=True, exist_ok=True)
invalid = sorted(set(PAPER_SAMPLE_NFE_VALUES) - set(NFE_VALUES))
if invalid:
    raise ValueError(f"PAPER_SAMPLE_NFE_VALUES contains invalid NFE values: {invalid}")

paper_seed = SEEDS[0]
paper_target = setting_2d.target.sample(SAMPLE_PLOT_N).cpu()
paper_endpoint = {}
paper_axis_arrays = [paper_target]

for nfe in PAPER_SAMPLE_NFE_VALUES:
    paper_endpoint[nfe] = {}
    for method in METHODS:
        samples = _load_or_generate_endpoint_samples(method, nfe, paper_seed, SAMPLE_PLOT_N)
        paper_endpoint[nfe][method] = samples
        paper_axis_arrays.append(samples)

paper_xmin, paper_xmax = _common_square_limits(paper_axis_arrays)

nrows = len(PAPER_SAMPLE_NFE_VALUES)
ncols = 1 + len(METHODS)
fig, axes = plt.subplots(
    nrows, ncols,
    figsize=(2.05 * ncols, 2.05 * nrows),
    squeeze=False, sharex=True, sharey=True,
)

rng = np.random.default_rng(2026)
for r, nfe in enumerate(PAPER_SAMPLE_NFE_VALUES):
    row_panels = [paper_target] + [paper_endpoint[nfe][m] for m in METHODS]
    row_titles = ["Target"] + [METHOD_LABELS[m] for m in METHODS]
    for c, (samples, title) in enumerate(zip(row_panels, row_titles)):
        ax = axes[r, c]
        array = torch.as_tensor(samples).cpu().numpy()
        if len(array) > 1600:
            array = array[rng.choice(len(array), 1600, replace=False)]
        ax.scatter(array[:, 0], array[:, 1], s=5, alpha=0.28)
        ax.set_xlim(paper_xmin, paper_xmax)
        ax.set_ylim(paper_xmin, paper_xmax)
        ax.set_aspect("equal", adjustable="box")
        if r == 0:
            ax.set_title(title, fontsize=9)
        if c == 0:
            ax.set_ylabel(f"NFE={nfe}\ncount 2")
        elif r == nrows - 1:
            ax.set_ylabel("")
        if r == nrows - 1:
            ax.set_xlabel("count 1")

fig.tight_layout()
fig.savefig(paper_dir / "figure1_endpoint_samples.pdf", bbox_inches="tight")
fig.savefig(paper_dir / "figure1_endpoint_samples.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"shared endpoint range: [{paper_xmin:.1f}, {paper_xmax:.1f}] on both axes")

In [ ]:
# PAPER FIGURE 2 — intermediate/native generation paths for ALL methods
paper_dir = part1_output / "paper_outputs"
paper_dir.mkdir(parents=True, exist_ok=True)
paper_intermediate = _build_intermediate_samples(SEEDS[0], INTERMEDIATE_PLOT_N)
row_keys = ["reference"] + METHODS

paper_intermediate_arrays = [
    torch.as_tensor(x) for row in paper_intermediate.values() for x in row
]
paper_ixmin, paper_ixmax = _common_square_limits(paper_intermediate_arrays)

# Short paper-facing row labels live in a dedicated left margin rather than
# inside the first-column axes, so long method names never overlap the plots.
_paper_row_labels = {
    "reference": "Reference bridge",
    "count_flow_map": f"Count Flow Map\n(NFE={INTERMEDIATE_FLOW_NFE})",
    "count_fm_unit_jump": f"Count-FM unit jump\n({INTERMEDIATE_NATIVE_STEPS} steps)",
    "count_fm_binomial_tau_leap": f"Count-FM tau-leap\n({INTERMEDIATE_NATIVE_STEPS} steps)",
    "countsdiff": f"CountsDiff\n({INTERMEDIATE_NATIVE_STEPS} steps)",
    "d3pm": f"D3PM\n({INTERMEDIATE_NATIVE_STEPS} steps)",
    "discrete_flow_map": f"Discrete Flow Maps\n(NFE={INTERMEDIATE_FLOW_NFE}; argmax decode)",
}

nrows = len(row_keys)
ncols = len(INTERMEDIATE_PROGRESS_VALUES)
fig, axes = plt.subplots(
    nrows, ncols,
    figsize=(1.78 * ncols + 2.0, 1.70 * nrows),
    squeeze=False, sharex=True, sharey=True,
)

rng = np.random.default_rng(31415)
for r, method_key in enumerate(row_keys):
    for c, progress in enumerate(INTERMEDIATE_PROGRESS_VALUES):
        ax = axes[r, c]
        array = torch.as_tensor(paper_intermediate[method_key][c]).cpu().numpy()
        if len(array) > 1400:
            array = array[rng.choice(len(array), 1400, replace=False)]
        ax.scatter(array[:, 0], array[:, 1], s=4.5, alpha=0.27)
        ax.set_xlim(paper_ixmin, paper_ixmax)
        ax.set_ylim(paper_ixmin, paper_ixmax)
        ax.set_aspect("equal", adjustable="box")
        ax.tick_params(labelsize=7, pad=1)
        if r == 0:
            ax.set_title(f"q={progress:g}", fontsize=9)
        if r == nrows - 1:
            ax.set_xlabel("count 1", fontsize=8)

# Reserve a real label column on the left; do not use ax.set_ylabel(method_name).
fig.subplots_adjust(left=0.245, right=0.99, top=0.97, bottom=0.06, wspace=0.10, hspace=0.15)
fig.canvas.draw()
for r, method_key in enumerate(row_keys):
    bbox = axes[r, 0].get_position()
    y = 0.5 * (bbox.y0 + bbox.y1)
    fig.text(
        0.225, y, _paper_row_labels[method_key],
        ha="right", va="center", fontsize=8.3, linespacing=1.15,
    )
fig.supylabel("count 2", x=0.015, fontsize=8.5)

fig.savefig(paper_dir / "figure2_intermediate_samples.pdf", bbox_inches="tight")
fig.savefig(paper_dir / "figure2_intermediate_samples.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# PAPER FIGURE 3 — compact quality / efficiency curves
paper_dir = part1_output / "paper_outputs"
paper_dir.mkdir(parents=True, exist_ok=True)

paper_method_order = [METHOD_LABELS[m] for m in METHODS]
paper_summary = (
    metrics_2d.groupby(["method_label", "nfe"])
    [["tv", "w2", "runtime_seconds"]]
    .mean()
    .reset_index()
)

fig, axes = plt.subplots(1, 3, figsize=(10.2, 2.85))
panels = [
    ("nfe", "tv", "NFE", "TV ↓"),
    ("nfe", "w2", "NFE", r"$W_2$ ↓"),
    ("runtime_seconds", "w2", "generation time (s)", r"$W_2$ ↓"),
]

for ax, (xcol, ycol, xlabel, ylabel) in zip(axes, panels):
    for label in paper_method_order:
        group = paper_summary.loc[paper_summary["method_label"] == label].sort_values("nfe")
        ax.plot(
            group[xcol], group[ycol],
            marker="o", markersize=3.5, linewidth=1.2,
            label=label,
        )
    if xcol == "nfe":
        ax.set_xscale("log", base=2)
        ax.set_xticks(NFE_VALUES, labels=[str(v) for v in NFE_VALUES])
    else:
        ax.set_xscale("log")
    ax.set_xlabel(xlabel, fontsize=9)
    ax.set_ylabel(ylabel, fontsize=9)
    ax.tick_params(labelsize=8)
    ax.grid(alpha=0.20, linewidth=0.5)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles, labels,
    loc="lower center", bbox_to_anchor=(0.5, -0.10),
    ncol=3, frameon=False, fontsize=8,
)
fig.tight_layout(rect=(0, 0.10, 1, 1))
fig.savefig(paper_dir / "figure3_quality_efficiency.pdf", bbox_inches="tight")
fig.savefig(paper_dir / "figure3_quality_efficiency.png", dpi=300, bbox_inches="tight")
plt.show()


# Part 2 — $d=32$, high counts

Together with Part 2, this isolates the **count-scale effect at fixed dimension**. 


Cluster/script mode: `python scripts/run_simulation.py --part 3 --device cuda`. The notebook reuses the same output directory.

In [ ]:
PART3 = make_config("scale_32_high")
part3_output = ROOT / "outputs" / "simulation" / "part3_d32_high"
metrics_32_high = run_simulation_suite(
    ROOT, PART3, output_dir=part3_output, device=DEVICE, resume=RESUME, progress=True
)
summary_32_high = metrics_32_high.groupby(["method_label", "nfe"])[
    ["sliced_w2", "mmd2_rbf", "mean_relative_l1", "variance_relative_l1",
     "zero_fraction_l1", "correlation_fro_per_dim", "runtime_seconds", "samples_per_second", "parameters"]
].agg(["mean", "std"])
display(summary_32_high)
plot_quality_curves(metrics_32_high, exact_2d=False)
plot_runtime_curve(metrics_32_high, "sliced_w2")


In [ ]:
# PAPER TABLE — 32D high-count endpoint quality at selected operating points
paper_dir = part3_output / "paper_outputs"
paper_dir.mkdir(parents=True, exist_ok=True)

invalid = sorted(set(PAPER_TABLE_NFE_VALUES) - set(NFE_VALUES))
if invalid:
    raise ValueError(f"PAPER_TABLE_NFE_VALUES contains invalid NFE values: {invalid}")

_method_order = [METHOD_LABELS[m] for m in METHODS]
_metric_specs = [
    ("sliced_w2", "SW$_2$ ↓", lambda x: f"{x:.3f}"),
    ("mmd2_rbf", "MMD$^2_{RBF}$ ↓", lambda x: f"{x:.4f}"),
    ("runtime_seconds", "Time (s) ↓", lambda x: f"{x:.3f}"),
]

paper_table_32 = pd.DataFrame(index=_method_order)
for nfe in PAPER_TABLE_NFE_VALUES:
    block = metrics_32_high.loc[metrics_32_high["nfe"] == nfe]
    for metric, label, formatter in _metric_specs:
        summary = block.groupby("method_label")[metric].agg(["mean", "std"])
        paper_table_32[(f"NFE={nfe}", label)] = [
            f"{formatter(summary.loc[name, 'mean'])} ± {formatter(summary.loc[name, 'std'])}"
            for name in _method_order
        ]

paper_table_32.columns = pd.MultiIndex.from_tuples(paper_table_32.columns)
paper_table_32.index.name = "Method"
display(paper_table_32)

flat_table_32 = paper_table_32.copy()
flat_table_32.columns = [f"{a} | {b}" for a, b in flat_table_32.columns]
flat_table_32.to_csv(paper_dir / "table2_32high.csv")
(paper_dir / "table2_32high.tex").write_text(
    paper_table_32.to_latex(escape=False, multicolumn=True, multirow=True),
    encoding="utf-8",
)
print("saved:", paper_dir / "table2_32high.csv")
print("saved:", paper_dir / "table2_32high.tex")


In [ ]:
# PAPER FIGURE — 32D high-count quality / efficiency curves
paper_dir = part3_output / "paper_outputs"
paper_dir.mkdir(parents=True, exist_ok=True)

paper_method_order = [METHOD_LABELS[m] for m in METHODS]
paper_summary_32 = (
    metrics_32_high.groupby(["method_label", "nfe"])
    [["sliced_w2", "mmd2_rbf", "runtime_seconds"]]
    .mean()
    .reset_index()
)

fig, axes = plt.subplots(1, 3, figsize=(10.2, 2.85))
panels = [
    ("nfe", "sliced_w2", "NFE", "SW$_2$ ↓"),
    ("nfe", "mmd2_rbf", "NFE", "MMD$^2_{RBF}$ ↓"),
    ("runtime_seconds", "sliced_w2", "generation time (s)", "SW$_2$ ↓"),
]

for ax, (xcol, ycol, xlabel, ylabel) in zip(axes, panels):
    for label in paper_method_order:
        group = paper_summary_32.loc[paper_summary_32["method_label"] == label].sort_values("nfe")
        ax.plot(
            group[xcol], group[ycol],
            marker="o", markersize=3.5, linewidth=1.2,
            label=label,
        )
    if xcol == "nfe":
        ax.set_xscale("log", base=2)
        ax.set_xticks(NFE_VALUES, labels=[str(v) for v in NFE_VALUES])
    else:
        ax.set_xscale("log")
    ax.set_xlabel(xlabel, fontsize=9)
    ax.set_ylabel(ylabel, fontsize=9)
    ax.tick_params(labelsize=8)
    ax.grid(alpha=0.20, linewidth=0.5)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles, labels,
    loc="lower center", bbox_to_anchor=(0.5, -0.10),
    ncol=3, frameon=False, fontsize=8,
)
fig.tight_layout(rect=(0, 0.10, 1, 1))
fig.savefig(paper_dir / "figure4_32high_quality_efficiency.pdf", bbox_inches="tight")
fig.savefig(paper_dir / "figure4_32high_quality_efficiency.png", dpi=300, bbox_inches="tight")
plt.show()
print("saved:", paper_dir / "figure4_32high_quality_efficiency.pdf")
print("saved:", paper_dir / "figure4_32high_quality_efficiency.png")
